# ⚡ FastAPI Embeddings Service (FAISS + Transformers)
A Jupyter notebook template demonstrating how to build a lightweight embeddings and semantic search API using FastAPI, FAISS, and Transformers — all running interactively within Saturn Cloud.

## 🧠 Overview
This notebook walks you through building a FastAPI-based Embeddings Service that:
- Generates text embeddings using a Transformer model
- Stores them in a FAISS index for similarity search
- Exposes both embedding and search endpoints via FastAPI

You’ll be able to:
- Add texts to the API dynamically
- Perform semantic similarity queries
- Test everything live inside a notebook

This is perfect for quickly prototyping or demonstrating retrieval-based workflows on [Saturn Cloud](https://saturncloud.io/).

## ⚙️ 1. Install Dependencies

In [ ]:
!pip install torch transformers sentence-transformers faiss-cpu fastapi uvicorn[standard] pydantic requests numpy

## 🧩 2. Load Embedding Model and Initialize FAISS

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print('🔧 Loading embedding model...')
model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
embedding_dim = model.get_sentence_embedding_dimension()
print(f'✅ Model loaded — Embedding dimension: {embedding_dim}')

# Initialize FAISS (L2 distance)
index = faiss.IndexFlatL2(embedding_dim)
texts = []

## 🧠 3. Define Core Embedding and Search Functions

In [ ]:
def add_text(text: str):
    vector = model.encode([text])[0]
    index.add(np.array([vector]).astype('float32'))
    texts.append(text)
    return {'message': 'Text added successfully.', 'total_texts': len(texts)}

def search_texts(query: str, top_k: int = 3):
    if len(texts) == 0:
        return {'error': 'No texts in index. Please add some first.'}

    query_vector = model.encode([query])[0]
    D, I = index.search(np.array([query_vector]).astype('float32'), top_k)
    results = [{'text': texts[i], 'distance': float(D[0][j])} for j, i in enumerate(I[0])]
    return {'query': query, 'results': results}

## ⚡ 4. Create the FastAPI Application

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title='FastAPI Embeddings Service (Notebook Edition)')

class TextIn(BaseModel):
    text: str

class SearchQuery(BaseModel):
    query: str
    top_k: int = 3

@app.post('/add_text')
def add_text_endpoint(item: TextIn):
    return add_text(item.text)

@app.post('/search')
def search_endpoint(query: SearchQuery):
    return search_texts(query.query, query.top_k)

@app.get('/healthz')
def healthz():
    return {'status': 'ok', 'count': len(texts)}

## 🌐 5. Run the API Server Inside Jupyter

In [ ]:
import threading, time, requests, uvicorn

PORT = 8002
_uvicorn_server = None
_uvicorn_thread = None

def start_api_in_thread(host='0.0.0.0', port=PORT, log_level='info'):
    global _uvicorn_server, _uvicorn_thread
    if _uvicorn_thread and _uvicorn_thread.is_alive():
        print(f'ℹ️ Server already running at http://127.0.0.1:{port}')
        return _uvicorn_thread

    config = uvicorn.Config(app, host=host, port=port, log_level=log_level)
    _uvicorn_server = uvicorn.Server(config)

    def _run():
        _uvicorn_server.run()

    _uvicorn_thread = threading.Thread(target=_run, daemon=True)
    _uvicorn_thread.start()

    for _ in range(30):
        try:
            time.sleep(0.1)
            r = requests.get(f'http://127.0.0.1:{port}/healthz', timeout=0.25)
            if r.status_code == 200:
                print(f'🚀 FastAPI running at http://127.0.0.1:{port} (thread: {_uvicorn_thread.name})')
                return _uvicorn_thread
        except Exception:
            pass
    print('⚠️ Server thread started but not reachable yet.')
    return _uvicorn_thread

def stop_api(join_timeout=5):
    global _uvicorn_server, _uvicorn_thread
    if _uvicorn_server is None or _uvicorn_thread is None:
        print('ℹ️ No server is currently running.')
        return
    _uvicorn_server.should_exit = True
    _uvicorn_thread.join(timeout=join_timeout)
    print('🛑 Server stopped.')
    _uvicorn_server = None
    _uvicorn_thread = None

## ▶️ 6. Start the FastAPI Service

In [ ]:
start_api_in_thread()

## 🧪 7a. Test the API

In [ ]:
import requests
requests.post('http://127.0.0.1:8002/add_text', json={'text': 'The quick brown fox jumps over the lazy dog.'}).json()
requests.post('http://127.0.0.1:8002/search', json={'query': 'A fast brown animal jumps over a sleepy dog', 'top_k': 3}).json()

## 🧪 7b. More Test the API (using more text)



In [ ]:
import requests

BASE_URL = "http://127.0.0.1:8002"

# --- Add multiple sample texts ---
samples = [
    "Artificial intelligence enables machines to learn from experience.",
    "Machine learning is a subset of artificial intelligence focused on pattern recognition.",
    "FastAPI is a modern, high-performance web framework for building APIs with Python.",
    "FAISS is a library for efficient similarity search and clustering of dense vectors.",
    "Deep learning models often require GPUs for accelerated computation.",
    "Natural language processing helps computers understand human language."
]

for text in samples:
    res = requests.post(f"{BASE_URL}/add_text", json={"text": text})
    print(f"📘 Added: {text[:50]}... -> {res.status_code}")

# --- Example 1: Semantic similarity query ---
query_1 = "What library is used for fast vector similarity search?"
result_1 = requests.post(f"{BASE_URL}/search", json={"query": query_1, "top_k": 3}).json()
print(f"\n🔍 Query: {query_1}")
print(result_1)

# --- Example 2: Conceptual link query ---
query_2 = "How do computers understand human speech?"
result_2 = requests.post(f"{BASE_URL}/search", json={"query": query_2, "top_k": 3}).json()
print(f"\n🔍 Query: {query_2}")
print(result_2)

# --- Example 3: Broader topic search ---
query_3 = "Explain how AI learns from data"
result_3 = requests.post(f"{BASE_URL}/search", json={"query": query_3, "top_k": 3}).json()
print(f"\n🔍 Query: {query_3}")
print(result_3)


## ⏹️ 8. Stop the API

In [ ]:
stop_api()

## 🏁 **Conclusion**

You’ve built a lightweight **FastAPI Embeddings Service** that generates and searches text embeddings using **Transformers** and **FAISS** — all within **Saturn Cloud**.

This template serves as a quick starting point for developing AI-powered APIs and retrieval systems.
You can extend it to support larger datasets, custom models, or integrate it into RAG pipelines directly in your Saturn workspace.

**Built with ❤️ using**
🤗 **Transformers** | 🧮 **FAISS** | ⚡ **FastAPI** | ☁️ **[Saturn Cloud](https://saturncloud.io/)**
